# WoT Discovery Retrieval — Benchmark & Examples

Interactive harness over the new evaluation library. Replaces the ad-hoc accuracy
counting in `benchmarking.ipynb` with the shared metrics in `eval_lib.py`:
Hit@k, Recall@k, Precision@k, MRR, no-answer AUROC/F1, and latency percentiles,
scored over the categorized query set in `queries.py`.

Run from the `code/` directory. Swap in the paper models (mpnet bi-encoders,
MiniLM-L6/L12 cross-encoders) where noted for the full run; the defaults use
small models so the notebook runs quickly.

In [1]:
import os, sys
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
sys.path.insert(0, '.')

import pandas as pd
from eval_lib import load_records, build_record_text, evaluate
from queries import QUERIES, category_counts, validate
from retrievers import BiEncoderRetriever

pd.set_option('display.max_colwidth', 90)
validate()
CSV = 'mainSimulationAccessTraces.csv'
records = build_record_text(load_records(CSV), fmt='sentence')
print('indexed records:', len(records))
print('query categories:', category_counts())
records.head()

indexed records: 286
query categories: {'templated': 12, 'paraphrased': 16, 'synonym': 12, 'abstract': 10, 'ambiguous_location': 6, 'ambiguous_device': 5, 'no_answer': 12}


,operation,destinationServiceType,destinationLocation,accessedNodeAddress,record_text
0,registerService,/lightControler,BedroomParents,/agent2/lightcontrol2,i need to register service light controler in bed room parents
1,registerService,/lightControler,Dinningroom,/agent3/lightcontrol3,i need to register service light controler in dinning room
2,registerService,/lightControler,BedroomChildren,/agent1/lightcontrol1,i need to register service light controler in bed room children
3,registerService,/lightControler,Kitchen,/agent4/lightcontrol4,i need to register service light controler in kitchen
4,registerService,/movementSensor,Kitchen,/agent4/movement4,i need to register service movement sensor in kitchen


## 1. Build a retriever
Frozen bi-encoder, optionally with cross-encoder reranking. The index is built once here.

In [2]:
retriever = BiEncoderRetriever(
    records,
    bi_encoder_name='sentence-transformers/all-MiniLM-L6-v2',
    # paper bi-encoders: 'sentence-transformers/all-mpnet-base-v2',
    #                    'sentence-transformers/multi-qa-mpnet-base-dot-v1'
    cross_encoder_name='cross-encoder/ms-marco-MiniLM-L-6-v2',
    use_reranker=False,   # set True to enable reranking
    device='cpu',
)
retriever.describe()

c:\dev\audacity_private\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9364.72it/s]


{'retriever': 'bi-encoder',
 'bi_encoder': 'all-MiniLM-L6-v2',
 'reranker': 'none',
 'load_time_s': 3.775,
 'index_build_time_s': 0.302,
 'embedding_mb': 0.419,
 'n_records': 286}

## 2. Run the benchmark
Overall metrics (over answerable queries), per-category breakdown, no-answer separability, and latency.

In [3]:
def print_report(res, title=''):
    o, na, lat = res['overall'], res['no_answer'], res['latency']
    if title:
        print('###', title)
    print(f"n_answerable={o['n']}")
    print('overall: ' + '  '.join(f'{k}={o[k]:.3f}' for k in o if k != 'n'))
    print(f"no-answer: AUROC={na['auroc']:.3f}  best-F1={na['best_f1']:.3f}  thr={na['best_f1_threshold']:.3f}")
    print(f"latency: mean={lat['mean']*1000:.1f}ms  p50={lat['p50']*1000:.1f}ms  p95={lat['p95']*1000:.1f}ms")
    cat = pd.DataFrame(res['by_category']).T
    keep = [c for c in ['n', 'hit@1', 'hit@3', 'hit@5', 'mrr'] if c in cat.columns]
    return cat[keep]

res = evaluate(retriever, QUERIES, ks=(1, 3, 5, 10))
print_report(res, retriever.describe()['bi_encoder'])

### all-MiniLM-L6-v2
n_answerable=61
overall: hit@1=0.787  hit@3=0.918  hit@5=0.918  hit@10=0.967  recall@1=0.645  recall@3=0.805  recall@5=0.841  recall@10=0.900  precision@1=0.787  precision@3=0.361  precision@5=0.261  precision@10=0.212  mrr=0.857
no-answer: AUROC=0.842  best-F1=0.917  thr=0.234
latency: mean=6.5ms  p50=5.9ms  p95=9.7ms


,n,hit@1,hit@3,hit@5,mrr
abstract,10.0,0.600000,0.900000,0.900000,0.744444
ambiguous_device,5.0,0.800000,0.800000,0.800000,0.800000
ambiguous_location,6.0,0.833333,0.833333,0.833333,0.861111
paraphrased,16.0,0.812500,0.875000,0.875000,0.854167
synonym,12.0,0.666667,1.000000,1.000000,0.833333
templated,12.0,1.000000,1.000000,1.000000,1.000000


## 3. Inspect per-query results and failures
Find the queries the retriever misses, and slice by category.

In [4]:
pq = res['per_query']
misses = pq[(pq.answerable) & (pq['hit@5'] == 0)]
print(f'{len(misses)} answerable queries missed at top-5')
display(misses[['id', 'category', 'top1_endpoint', 'top_score']])

# slice one category (change as desired)
pq[pq.category == 'paraphrased'][['id', 'hit@1', 'hit@3', 'mrr', 'top1_endpoint']]

5 answerable queries missed at top-5


,id,category,top1_endpoint,top_score
17,para06,paraphrased,/agent25/heatingcontrol4,0.435900
25,para14,paraphrased,/agent25/heatingcontrol4,0.488864
46,abs07,abstract,/agent11/heatingcontrol2,0.234221
51,ambl02,ambiguous_location,/agent25/heatingcontrol4,0.431377
60,ambd05,ambiguous_device,/agent3/heatingcontrol1,0.355478


,id,hit@1,hit@3,mrr,top1_endpoint
12,para01,1.0,1.0,1.000000,/agent2/lightcontrol2/lightOn
13,para02,1.0,1.0,1.000000,/agent3/lightcontrol3/lightOn
14,para03,1.0,1.0,1.000000,/agent1/lightcontrol1/lightOn
15,para04,1.0,1.0,1.000000,/agent4/lightcontrol4/lightOn
16,para05,1.0,1.0,1.000000,/agent4/movement4/movement
17,para06,0.0,0.0,0.166667,/agent25/heatingcontrol4
18,para07,0.0,1.0,0.500000,/agent2/movement2/lastChange
19,para08,1.0,1.0,1.000000,/agent4/tempin4
20,para09,1.0,1.0,1.000000,/agent1/tempin1
21,para10,1.0,1.0,1.000000,/agent1/movement1/movement


## 4. Example explorer
Type any query and see the ranked endpoints with scores (like the ad-hoc examples before).

In [5]:
def show_query(q, top_k=5):
    print('QUERY:', q)
    for ep, sc in retriever.search(q, top_k):
        print(f'  {sc:8.3f}  {ep}')
    print()

show_query('how warm is it in the kitchen?')
show_query('turn on the light')          # ambiguous device
show_query('show me the front-door camera')  # no-answer
show_query('is anyone home?')            # abstract

QUERY: how warm is it in the kitchen?
     0.307  /agent3/heatingcontrol1
     0.300  /agent4/tempin4
     0.281  /agent3/heatingcontrol1/heatingOn
     0.279  /agent11/heatingcontrol2
     0.278  /agent4/movement4/movement

QUERY: turn on the light
     0.452  /agent10/lightcontrol10/lightOn
     0.437  /agent10/lightcontrol10/lightOn
     0.437  /agent10/lightcontrol10
     0.436  /agent2/lightcontrol2/lightOn
     0.428  /agent29/lightcontrol29/lightOn

QUERY: show me the front-door camera
     0.356  /agent12/doorlock3/open
     0.355  /agent12/lightcontrol12/lightOn
     0.348  /agent3/doorlock1/open
     0.346  /agent10/doorlock2/open
     0.342  /agent3/lightcontrol3/lightOn

QUERY: is anyone home?
     0.169  /agent10/tempin10
     0.168  /agent10/tempin10
     0.156  /agent23/tempin23
     0.150  /agent23/tempin23
     0.149  /agent24/tempin24



## 5. Combinatorial sweep
Compares bi-encoder x cross-encoder x record-format x reranking with the new metrics
(the modern replacement for the old `run_combinatorial_benchmarks`). Add the paper
models to `bi_list` / `cross_list` for the full run.

In [6]:
def run_sweep(csv, bi_list, cross_list, formats=('sentence',),
              rerank=(False, True), ks=(1, 3, 5, 10)):
    base = load_records(csv)
    rows = []
    for fmt in formats:
        recs = build_record_text(base, fmt=fmt)
        for bi in bi_list:
            for cross in cross_list:
                for rr in rerank:
                    r = BiEncoderRetriever(recs, bi, cross, use_reranker=rr, device='cpu')
                    res = evaluate(r, QUERIES, ks=ks)
                    o, na, lat = res['overall'], res['no_answer'], res['latency']
                    rows.append({
                        'format': fmt, 'bi': r.name, 'cross': (r.cross_name or '-'),
                        'rerank': rr,
                        'hit@1': round(o['hit@1'], 3), 'hit@3': round(o['hit@3'], 3),
                        'hit@5': round(o['hit@5'], 3), 'mrr': round(o['mrr'], 3),
                        'na_auroc': round(na['auroc'], 3),
                        'p95_ms': round(lat['p95'] * 1000, 1),
                    })
    return pd.DataFrame(rows)

sweep = run_sweep(
    CSV,
    bi_list=['sentence-transformers/all-MiniLM-L6-v2'],
    cross_list=['cross-encoder/ms-marco-MiniLM-L-6-v2'],
    formats=('sentence', 'tuple'),
    rerank=(False, True),
)
sweep

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5051.23it/s]


,format,bi,cross,rerank,hit@1,hit@3,hit@5,mrr,na_auroc,p95_ms
0,sentence,all-MiniLM-L6-v2,-,False,0.787,0.918,0.918,0.857,0.842,6.6
1,sentence,all-MiniLM-L6-v2,ms-marco-MiniLM-L-6-v2,True,0.672,0.918,0.934,0.798,0.832,79.9
2,tuple,all-MiniLM-L6-v2,-,False,0.738,0.885,0.918,0.819,0.816,8.1
3,tuple,all-MiniLM-L6-v2,ms-marco-MiniLM-L-6-v2,True,0.689,0.918,0.951,0.805,0.769,88.7
